In [40]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sentence_transformers import SentenceTransformer
import torch

In [41]:
train_dataset = pd.read_csv("C:/Users/sevinj.rahimova/Desktop/Kaggle competition/RSNA Knee Abnormality Detection/Data/train.csv")

In [42]:
train_dataset.head()

,StudyInstanceUID,Report,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,1.2.826.0.1.3680043.8.498.10004873229099053869...,Técnica: RMN de la rodilla. Resultados: Rotura...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1.2.826.0.1.3680043.8.498.10004945927472656027...,[DATE]: * MR Knie Rechts 15ch AA Klinische Inl...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1.2.826.0.1.3680043.8.498.10009278692606631573...,Hallazgos:\nNo hay alteraciones en significati...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1.2.826.0.1.3680043.8.498.10009639203170750274...,"In the medial compartment, the meniscus is no...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1.2.826.0.1.3680043.8.498.10013663742400736029...,CONSTATATIONS :\n\nFractures :\nAucune.\n\nAli...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [43]:
def basic_eda(train_dataset):
   
    print("--- Dataset Shape (Rows, Columns) ---")
    print(train_dataset.shape, "\n")
    
    print("--- Data Types & Non-Null Counts ---")
    print(train_dataset.info(), "\n")
    print(" ")
    print("--- Missing Values Count ---")
    print(train_dataset.isnull().sum(), "\n")
    print(" ")
    print("--- Duplicate Rows Count ---")
    print(f"Duplicates: {train_dataset.duplicated().sum()}\n")
    print(" ")
    print("--- Descriptive Statistics (Numeric) ---")
    print(train_dataset.describe(), "\n")
    print(" ")
    print("--- Descriptive Statistics (Categorical) ---")
    print(train_dataset.describe(include=['O']), "\n")


basic_eda(train_dataset)

--- Dataset Shape (Rows, Columns) ---
(4407, 14) 

--- Data Types & Non-Null Counts ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4407 entries, 0 to 4406
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   StudyInstanceUID  4407 non-null   object 
 1   Report            4407 non-null   object 
 2   ACL               58 non-null     float64
 3   MCL               58 non-null     float64
 4   Medial Meniscus   58 non-null     float64
 5   Lateral Meniscus  58 non-null     float64
 6   Medial OA         58 non-null     float64
 7   Lateral OA        58 non-null     float64
 8   PF OA             58 non-null     float64
 9   Effusion          58 non-null     float64
 10  Synovitis         58 non-null     float64
 11  Baker's           58 non-null     float64
 12  Contusion         58 non-null     float64
 13  Fracture          58 non-null     float64
dtypes: float64(12), object(2)
memory usage: 482.1+ K

In [44]:
train_dataset.nunique()

StudyInstanceUID    4407
Report              4276
ACL                    2
MCL                    2
Medial Meniscus        2
Lateral Meniscus       2
Medial OA              2
Lateral OA             2
PF OA                  2
Effusion               2
Synovitis              2
Baker's                2
Contusion              2
Fracture               2
dtype: int64

In [45]:
train_dataset["ACL"].unique()

array([nan,  0.,  1.])

In [46]:
# Label distribution: Positive (1), Negative (0), Missing (NaN)

label_columns = [
    'ACL',
    'MCL',
    'Medial Meniscus',
    'Lateral Meniscus',
    'Medial OA',
    'Lateral OA',
    'PF OA',
    'Effusion',
    'Synovitis',
    "Baker's",
    'Contusion',
    'Fracture'
]

distribution = pd.DataFrame({
    'Positive (1)': [train_dataset[col].eq(1).sum() for col in label_columns],
    'Negative (0)': [train_dataset[col].eq(0).sum() for col in label_columns],
    'Missing (NaN)': [train_dataset[col].isna().sum() for col in label_columns]
}, index=label_columns)

print(distribution)

                  Positive (1)  Negative (0)  Missing (NaN)
ACL                         24            34           4349
MCL                          9            49           4349
Medial Meniscus             26            32           4349
Lateral Meniscus            23            35           4349
Medial OA                   15            43           4349
Lateral OA                  11            47           4349
PF OA                       21            37           4349
Effusion                    35            23           4349
Synovitis                   27            31           4349
Baker's                     12            46           4349
Contusion                   19            39           4349
Fracture                    18            40           4349


In [47]:
imbalance_table = pd.DataFrame({
    'Positive': [train_dataset[col].eq(1).sum() for col in label_columns],
    'Negative': [train_dataset[col].eq(0).sum() for col in label_columns]
}, index=label_columns)

imbalance_table['Positive %'] = (
    imbalance_table['Positive'] /
    (imbalance_table['Positive'] + imbalance_table['Negative'])
) * 100

imbalance_table['Negative %'] = (
    imbalance_table['Negative'] /
    (imbalance_table['Positive'] + imbalance_table['Negative'])
) * 100

imbalance_table['Imbalance Ratio'] = (
    imbalance_table['Negative'] /
    imbalance_table['Positive']
)

imbalance_table = imbalance_table.sort_values(
    'Imbalance Ratio',
    ascending=False
)

print(imbalance_table.round(2))

                  Positive  Negative  Positive %  Negative %  Imbalance Ratio
MCL                      9        49       15.52       84.48             5.44
Lateral OA              11        47       18.97       81.03             4.27
Baker's                 12        46       20.69       79.31             3.83
Medial OA               15        43       25.86       74.14             2.87
Fracture                18        40       31.03       68.97             2.22
Contusion               19        39       32.76       67.24             2.05
PF OA                   21        37       36.21       63.79             1.76
Lateral Meniscus        23        35       39.66       60.34             1.52
ACL                     24        34       41.38       58.62             1.42
Medial Meniscus         26        32       44.83       55.17             1.23
Synovitis               27        31       46.55       53.45             1.15
Effusion                35        23       60.34       39.66    

In [73]:

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Yalnız label-i məlum olan MCL nümunələri
y_true = train_dataset['MCL'].dropna().astype(int)

# Sadə baseline:
# model bütün xəstələrə "negative" (0) deyir
y_pred = np.zeros(len(y_true), dtype=int)

In [74]:
print("y_true:", y_true.shape)
print("y_pred:", y_pred.shape)

y_true: (58,)
y_pred: (58,)


In [50]:
cm = confusion_matrix(y_true, y_pred)

print(cm)

[[49  0]
 [ 9  0]]


In [51]:
accuracy = accuracy_score(y_true, y_pred)

precision = precision_score(
    y_true,
    y_pred,
    zero_division=0
)

recall = recall_score(
    y_true,
    y_pred,
    zero_division=0
)

f1 = f1_score(
    y_true,
    y_pred,
    zero_division=0
)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

Accuracy : 0.8448
Precision: 0.0000
Recall   : 0.0000
F1 Score : 0.0000


In [52]:
df_mcl = train_dataset[
    train_dataset['MCL'].notna()
][['Report', 'MCL']].copy()

df_mcl['MCL'] = df_mcl['MCL'].astype(int)

print(df_mcl.shape)
print(df_mcl['MCL'].value_counts())

(58, 2)
MCL
0    49
1     9
Name: count, dtype: int64


In [53]:
from sklearn.model_selection import train_test_split

X = df_mcl['Report']
y = df_mcl['MCL']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test :", X_test.shape)

print("\nTrain distribution:")
print(y_train.value_counts())

print("\nTest distribution:")
print(y_test.value_counts())

Train: (46,)
Test : (12,)

Train distribution:
MCL
0    39
1     7
Name: count, dtype: int64

Test distribution:
MCL
0    10
1     2
Name: count, dtype: int64


In [54]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2)
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Train TF-IDF shape:", X_train_tfidf.shape)
print("Test TF-IDF shape :", X_test_tfidf.shape)

Train TF-IDF shape: (46, 5000)
Test TF-IDF shape : (12, 5000)


In [55]:
tfidf.fit_transform(X_train)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 8837 stored elements and shape (46, 5000)>

In [56]:
tfidf.transform(X_test)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 2155 stored elements and shape (12, 5000)>

In [57]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    random_state=42
)

model.fit(X_train_tfidf, y_train)

,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following a

In [58]:
y_pred = model.predict(X_test_tfidf)

In [59]:
y_prob = model.predict_proba(X_test_tfidf)[:, 1]

In [60]:
print(y_pred)

[0 0 0 0 0 0 0 0 0 0 0 0]


In [61]:
print(y_prob)

[0.1246718  0.13522934 0.15492451 0.11223931 0.1552786  0.16823008
 0.13681901 0.19728761 0.14863558 0.14145638 0.17869751 0.14430204]


In [62]:
results = pd.DataFrame({
    'Actual': y_test.values,
    'Probability': y_prob
})

results = results.sort_values(
    'Probability',
    ascending=False
)

print(results)

    Actual  Probability
7        0     0.197288
10       0     0.178698
5        0     0.168230
4        0     0.155279
2        0     0.154925
8        1     0.148636
11       0     0.144302
9        0     0.141456
6        1     0.136819
1        0     0.135229
0        0     0.124672
3        0     0.112239


In [63]:
model_balanced = LogisticRegression(
    class_weight='balanced',
    random_state=42
)

model_balanced.fit(X_train_tfidf, y_train)

y_prob_balanced = model_balanced.predict_proba(
    X_test_tfidf
)[:, 1]

y_pred_balanced = (
    y_prob_balanced >= 0.5
).astype(int)

In [64]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test,
    y_pred_balanced,
    zero_division=0
))

              precision    recall  f1-score   support

           0       0.82      0.90      0.86        10
           1       0.00      0.00      0.00         2

    accuracy                           0.75        12
   macro avg       0.41      0.45      0.43        12
weighted avg       0.68      0.75      0.71        12



In [65]:
for threshold in [0.5, 0.4, 0.3, 0.2, 0.1]:
    
    y_pred_threshold = (
        y_prob_balanced >= threshold
    ).astype(int)
    
    print(f"\nThreshold = {threshold}")
    
    print(classification_report(
        y_test,
        y_pred_threshold,
        zero_division=0
    ))


Threshold = 0.5
              precision    recall  f1-score   support

           0       0.82      0.90      0.86        10
           1       0.00      0.00      0.00         2

    accuracy                           0.75        12
   macro avg       0.41      0.45      0.43        12
weighted avg       0.68      0.75      0.71        12


Threshold = 0.4
              precision    recall  f1-score   support

           0       0.83      0.50      0.62        10
           1       0.17      0.50      0.25         2

    accuracy                           0.50        12
   macro avg       0.50      0.50      0.44        12
weighted avg       0.72      0.50      0.56        12


Threshold = 0.3
              precision    recall  f1-score   support

           0       1.00      0.10      0.18        10
           1       0.18      1.00      0.31         2

    accuracy                           0.25        12
   macro avg       0.59      0.55      0.24        12
weighted avg       0.86

-------------------------

In [66]:
train_dataset['Report'].sample(20, random_state=42).tolist()

['Regelrechte Stellungsverhältnisse im Kniegelenk. Allseits regelrechtes Knochenmarksignal. Kein Hinweis auf eine Fraktur oder ein Bone-bruise. Stigmata der medial betonten Gonarthrose mit Gelenkspaltverschmälerung und osteophytären Randausziehungen sowie Chondropathie femorotibial bds. Keine höhergradige Chondropathie femoropatellar. Vorderes und hinteres Kreuzband sowie LCL intakt. Quadrizeps- und Patellarsehne intakt ohne Signalalteration. Der Innenmeniskus mukoide mit horizontaler Rissbildung. Auch der Außenmeniskus mukoid verändert ohne Nachweis einer Rissbildung. Fibroostosen der Quadrizepssehne. Geringer Gelenkerguss. Unauffällige Darstellung der Weichteile.',
 'SAĞ DİZ MRG. Tetkik protokolü: Çok düzlemli, çok sekanslı. Bulgular: Medial meniscüs posterior hornunda grade II dejenerasyon ve ayrıca horizontal yırtık ile uyumlu görünüm mevcuttur. Medial menisküs posterior horn komşuluğunda 7x6 mm boyutlarında ince septalı parameniskal kist izlendi.Lateral menisküste anterior horn gr

In [67]:
acl_data = train_dataset[
    train_dataset['ACL'].notna()
][['Report', 'ACL']].copy()

acl_data['ACL'] = acl_data['ACL'].astype(int)

In [68]:
for _, row in acl_data.iterrows():
    print("=" * 100)
    print("LABEL:", row["ACL"])
    print(row["Report"])

LABEL: 0
Antecedentes Clínicos:
Esguince rodilla. [DATE].
Hallazgos:
No hay alteraciones de señal significativas de la médula ósea.
Ligamentos cruzados y colaterales dentro de límites normales.
Amputación marginal del cuerpo del menisco lateral. Menisco medial de morfología y señal
conservada, sin signos de rotura.
Cartílagos de los compartimentos femorotibiales sin alteraciones.
Fina úlcera condral focal de espesor total del aspecto inferior de la vertiente medial de la tróclea
femoral con mínimos cambios óseos secundarios. Fenómenos condrales reparativos de la región
central de la simple femoral. Cartílago rotuliano sin alteraciones.
Leve derrame articular. No hay quistes poplíteos patológicos.
Aumento de señal de la inserción distal del tendón cuadricipital y de la inserción proximal del
tendón rotuliano, sin signos de rotura.
No hay alteraciones de señal de la grasa de Hoffa.
Impresión:
Amputación marginal del cuerpo del menisco lateral.
Condropatía focal grado 4 del aspecto inferi

In [69]:
# Hesabatlardakı ümumi strukturu və əsas terminlərin paylanmasını yoxlayırıq
reports = train_dataset['Report'].dropna()
print(f'Ümumi hesabat sayı: {len(reports)}')
print(f'Unikal hesabat sayı: {reports.nunique()}')

# Məsələn, "intact" və "tear" sözlərinin keçmə tezliyi
intact_count = reports.str.contains('intact|normal|preserved', case=False).sum()
tear_count = reports.str.contains('tear|rupture|rotura|ρήξη', case=False).sum()

print(f'"Intact/Normal" keçən hesabat sayı: {intact_count}')
print(f'"Tear/Rupture" keçən hesabat sayı: {tear_count}')

Ümumi hesabat sayı: 4407
Unikal hesabat sayı: 4276
"Intact/Normal" keçən hesabat sayı: 2753
"Tear/Rupture" keçən hesabat sayı: 2500


In [70]:
# Hər bir hədəf sütunu üzrə positive (1) və negative (0) saylarını yoxlayırıq
target_columns = [
    'ACL',
    'MCL',
    'Medial Meniscus',
    'Lateral Meniscus',
    'Medial OA',
    'Lateral OA',
    'PF OA',
    'Effusion',
    'Synovitis',
    "Baker's",
    'Contusion',
    'Fracture',
]
label_counts = train_dataset[target_columns].sum()
print("Hər hədəf üzrə müsbət (1) halların sayı:")
print(label_counts)

Hər hədəf üzrə müsbət (1) halların sayı:
ACL                 24.0
MCL                  9.0
Medial Meniscus     26.0
Lateral Meniscus    23.0
Medial OA           15.0
Lateral OA          11.0
PF OA               21.0
Effusion            35.0
Synovitis           27.0
Baker's             12.0
Contusion           19.0
Fracture            18.0
dtype: float64


In [71]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.pipeline import Pipeline



# 1(Keywords)

keywords_expanded = {
    'ACL': [
        'acl',
        'anterior cruciate',
        'vorderes kreuzband',
        'cruzado anterior',
        'ön çapraz',
        'kreuzband',
        'anterior cruciate ligament',
        'vkb',
        'lkd',
        'lca',
        'χιαστός',
    ],
    'MCL': [
        'mcl',
        'medial collateral',
        'innenband',
        'colateral medial',
        'iç yan',
        'medial collateral ligament',
        'lcm',
        'medial kollateral',
        'mcm',
    ],
    'Medial Meniscus': [
        'medial meniscus',
        'menisco medial',
        'innenmeniskus',
        'medyal menisküs',
        'medial menisküs',
        'posterior horn medial meniscus',
        'meniscal tear medial',
        'menisco interno',
        'έσω μηνίσκος',
        'iç menisküs',
    ],
    'Lateral Meniscus': [
        'lateral meniscus',
        'menisco lateral',
        'außenmeniskus',
        'lateral menisküs',
        'anterior horn lateral meniscus',
        'menisco externo',
        'έξω μηνίσκος',
        'dış menisküs',
    ],
    'Medial OA': [
        'medial oa',
        'medial osteoarthritis',
        'artrosis medial',
        'femorotibial medial',
        'chondrosis medial',
        'medial compartment degeneration',
        'gonavartrose medial',
        'gonartrose',
        'medial tibiofemoral compartment',
        'medial compartment chondrosis',
        'medial joint space',
        'chondromalacia medial',
        'cartilage loss medial',
        'έσω',
        'χόνδρου',
        'medial tibial plateau',
        'subchondral',
        'osteophyte',
        'osteofytose',
        'kraakbeenverlies',
        'medial femorotibial',
        'medial kompartment',
    ],
    'Lateral OA': [
        'lateral oa',
        'lateral osteoarthritis',
        'artrosis lateral',
        'femorotibial lateral',
        'chondrosis lateral',
        'lateral compartment degeneration',
        'lateraal',
        'lateral femorotibiaal',
        'laterale tibiaplateau',
        'lateral compartment chondrosis',
        'kraakbeenlijden lateraal',
        'lateral joint space narrowing',
        'lateral kompartment',
    ],
    'PF OA': [
        'pf oa',
        'patellofemoral',
        'retropatellar',
        'chondromalacia patellae',
        'patella',
        'trochlea',
        'condropatía rotuliana',
        'condropatia rotuliana',
        'úlcera condral',
        'femoropatelar',
        'artrosis femoropatelar',
        'patellofemoral chondropathy',
        'kondromalazi',
        'troklear',
    ],
    'Effusion': [
        'effusion',
        'derrame',
        'erguss',
        'fluid',
        'líquido',
        'sıvı',
        'gelenkflüssigkeit',
        'joint effusion',
        'hydrops',
        'efüzyon',
        'izlıv',
        'su toplama',
        'fluid collection',
        'fluid accumulation',
        'joint fluid',
        'liquid collection',
        'synovial fluid',
    ],
    'Synovitis': [
        'synovitis',
        'sinovitis',
        'synovial',
        'synoviale',
        'sinovit',
        'synovial thickening',
        'reizsynovialitis',
        'synovial proliferation',
        'synovial membrane thickening',
        'hypertrophy of the synovium',
        'reizsinoviyalitis',
        'synovial hypertrophy',
    ],
    "Baker's": [
        'baker',
        'popliteal cyst',
        'quiste poplíteo',
        'bechter',
        'popliteal',
        'baker kisti',
        'baker cisti',
        'popliteal bursa',
    ],
    'Contusion': [
        'contusion',
        'contusión',
        'bone bruise',
        'ödemsignal',
        'ödem',
        'marrow edema',
        'bone marrow',
        'kontüzyon',
        'koştani edem',
        'botoedeem',
        'kemik iliği ödemi',
    ],
    'Fracture': [
        'fracture',
        'fractura',
        'fraktur',
        'kırık',
        'osseous defect',
        'bone fracture',
        'fraktür',
        'impactiefractuur',
        'subchondral fracture',
        'avulsion',
    ],
}

target_columns = list(keywords_expanded.keys())


# 2. Rule-Based Engine V2
def advanced_sentence_predict_v2(report):
  if pd.isna(report):
    return {col: 0 for col in target_columns}

  sentences = str(report).split('.')
  preds = {}

  for col in target_columns:
    col_matched = 0
    kws = keywords_expanded[col]

    for sentence in sentences:
      sent_lower = sentence.lower()
      if any(kw in sent_lower for kw in kws):
        if col in ['Effusion', 'Synovitis', "Baker's"]:
          has_negation = any(
              neg in sent_lower
              for neg in [
                  'no ',
                  'without ',
                  'free of',
                  'absent',
                  'not seen',
                  'no evidence',
              ]
          )
          if not has_negation:
            col_matched = 1
            break
        else:
      
          is_intact = any(
              neg in sent_lower
              for neg in ['intact', 'normal', 'preserved', 'within normal']
          )
          is_torn = any(
              pos in sent_lower
              for pos in [
                  'tear',
                  'rupture',
                  'lesion',
                  'fracture',
                  'edema',
                  'sprain',
                  'defect',
                  'degenerative',
              ]
          )

          if is_torn and not is_intact:
            col_matched = 1
            break
          elif is_intact and not is_torn:
            col_matched = 0
            break

    preds[col] = col_matched
  return preds


print('Rule-based engine is ready')


# 3. training process HyBRiD MODEL(TF-IDF + ML)

def train_hybrid_models(train_df):
  labeled_mask = train_df['ACL'].notna()
  df_gold = train_df[labeled_mask].copy()
  X_train = df_gold['Report'].astype(str)

  ml_models = {}
  print('ML modelləri öyrədilir...')

  for col in target_columns:
    y_train = df_gold[col].values.astype(int)
    if len(np.unique(y_train)) < 2:
      continue

    pipeline = Pipeline([
        (
            'tfidf',
            TfidfVectorizer(
                ngram_range=(1, 2), max_features=1000, stop_words='english'
            ),
        ),
        ('clf', LogisticRegression(class_weight='balanced', max_iter=1000)),
    ])
    pipeline.fit(X_train, y_train)
    ml_models[col] = pipeline

  print('TF-IDF nd Logistic Regression ')
  return ml_models, df_gold


# 4.hybrid predictor and ensemble

def create_hybrid_predictor(ml_models):
  def hybrid_predict(report):
    rule_preds = advanced_sentence_predict_v2(report)

    ml_preds = {}
    for col in target_columns:
      if col in ml_models:
        pred = ml_models[col].predict([str(report)])[0]
        ml_preds[col] = int(pred)
      else:
        ml_preds[col] = 0

    # Logical OR Ensemble 
    final_preds = {}
    for col in target_columns:
      r_val = rule_preds.get(col, 0)
      m_val = ml_preds.get(col, 0)
      final_preds[col] = 1 if (r_val == 1 or m_val == 1) else 0

    return final_preds

  return hybrid_predict


# 5.PIPELINE EXECUTION

ml_models, df_gold = train_hybrid_models(train_dataset)
predict_func = create_hybrid_predictor(ml_models)


all_predictions = []
for rep in train_dataset['Report']:
  preds = predict_func(rep)
  all_predictions.append(preds)

df_submission = pd.DataFrame(all_predictions)
if 'Id' in train_dataset.columns:
  df_submission.insert(0, 'Id', train_dataset['Id'])

df_submission.to_csv('submission.csv', index=False)
print('"submission.csv" file is ready.')
display(df_submission.head())

Rule-based engine is ready
ML modelləri öyrədilir...
TF-IDF nd Logistic Regression 
"submission.csv" file is ready.


,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,0,1,0,0,0,0,1,1,1,0,0,0
1,1,0,0,0,0,0,1,1,1,0,0,0
2,0,0,0,1,0,1,1,0,0,0,0,0
3,1,0,1,1,0,0,1,0,0,1,1,1
4,0,0,0,0,0,0,0,0,0,1,0,1


In [75]:
sample_subm=pd.read_csv("C:/Users/sevinj.rahimova/Desktop/Kaggle competition/RSNA Knee Abnormality Detection/sample_submission.csv")

In [76]:
sample_subm

,StudyInstanceUID,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,1.2.826.0.1.3680043.8.498.10047035057544427318...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
1,1.2.826.0.1.3680043.8.498.10062861783145312629...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
2,1.2.826.0.1.3680043.8.498.10067514707072572280...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
